In [ ]:
!cd /content/

In [ ]:
!rm -rf llms/

In [4]:
!git clone https://github.com/seshuthota/llms.git

Cloning into 'llms'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 34 (delta 16), reused 25 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 11.69 KiB | 5.85 MiB/s, done.
Resolving deltas: 100% (16/16), done.


In [ ]:
%cd llms

/content/llms


In [ ]:
!pip install torch tiktoken datasets

In [ ]:
# Fetch Hugging Face token required as environmental variable for training script
import os
hf_token = None

# Check if running in Kaggle
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN successfully loaded from Kaggle Secrets!")
except ImportError:
    pass

# Check if running in Google Colab
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
        print("HF_TOKEN successfully loaded from Colab Secrets!")
    except ImportError:
        pass

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
else:
    print("WARNING: HF_TOKEN not found in environments. Make sure you set it in your environment secrets.")


In [ ]:
!torchrun --nproc_per_node=2 train.py

2026-03-03 10:53:53,118 - INFO - ==================================================
2026-03-03 10:53:53,118 - INFO - Starting training pipeline
2026-03-03 10:53:53,118 - INFO - ==================================================
2026-03-03 10:53:53,118 - INFO - Loading TinyStories dataset from HuggingFace...
2026-03-03 10:53:53,412 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-03 10:53:53,419 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/roneneldan/TinyStories/f54c09fd23315a6f9c86f9dc80f725de7d8f9c64/README.md "HTTP/1.1 200 OK"
2026-03-03 10:53:53,646 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/roneneldan/TinyStories/resolve/f54c09fd23315a6f9c86f9dc80f725de7d8f9c64/TinyStories.py "HTTP/1.1 404 Not Found"
2026-03-03 10:53:54,325 - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/roneneldan/TinySto

In [ ]:
!ls -lrt

total 636804
-rw-r--r-- 1 root root      2899 Mar  3 10:53 attention.py
-rw-r--r-- 1 root root      3081 Mar  3 10:53 transformer.py
-rw-r--r-- 1 root root      7406 Mar  3 10:53 train.py
-rw-r--r-- 1 root root      1114 Mar  3 10:53 tokenizer.py
-rw-r--r-- 1 root root      1666 Mar  3 10:53 model.py
-rw-r--r-- 1 root root      1914 Mar  3 10:53 generate.py
-rw-r--r-- 1 root root       994 Mar  3 10:53 embeddings.py
-rw-r--r-- 1 root root      6490 Mar  3 10:53 dataset.py
drwxr-xr-x 2 root root      4096 Mar  3 10:53 __pycache__
drwxr-xr-x 2 root root      4096 Mar  3 10:53 logs
drwxr-xr-x 2 root root      4096 Mar  3 12:00 checkpoints
-rw-r--r-- 1 root root 652028747 Mar  3 12:02 gpt_model.pt


In [ ]:
!pip install huggingface-hub

In [ ]:
from huggingface_hub import HfApi, login

In [ ]:
api = HfApi()

In [ ]:
user_info = api.whoami()
print(f"Logged in as: {user_info['name']}")
print(f"Email: {user_info.get('email', 'N/A')}")

Logged in as: CuriousDragon
Email: curiousechoes27@gmail.com


: 

: 

In [ ]:
repo_id = "CuriousDragon/gpt-tinystories"
api.create_repo(repo_id, repo_type="model", exist_ok=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


RepositoryNotFoundError: 404 Client Error. (Request ID: Root=1-69a6ce35-3b09265b55301f577399691d;80408adf-6407-430d-b051-c0e31f94d515)

Repository Not Found for url: https://huggingface.co/api/models/CuriousDragon/gpt-tinystories/preupload/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Note: Creating a commit assumes that the repo already exists on the Huggingface Hub. Please use `create_repo` if it's not the case.

In [ ]:
# Then upload
api.upload_file(
    path_or_fileobj="gpt_model.pt",
    path_in_repo="pytorch_model.pt",
    repo_id=repo_id,
    repo_type="model"
)
print(f"Uploaded to https://huggingface.co/{repo_id}")